In [1]:
import re
from collections import Counter
from sklearn.model_selection import train_test_split

### Data Preprocessing

In [2]:
file_path = "rus.txt"
pairs = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            eng = parts[0]
            rus = parts[1]
            pairs.append((eng, rus))


print(pairs[:20])
print(len(pairs))

[('Go.', 'Марш!'), ('Go.', 'Иди.'), ('Go.', 'Идите.'), ('Hi.', 'Здравствуйте.'), ('Hi.', 'Привет!'), ('Hi.', 'Хай.'), ('Hi.', 'Здрасте.'), ('Hi.', 'Здоро́во!'), ('Run!', 'Беги!'), ('Run!', 'Бегите!'), ('Run.', 'Беги!'), ('Run.', 'Бегите!'), ('Who?', 'Кто?'), ('Wow!', 'Вот это да!'), ('Wow!', 'Круто!'), ('Wow!', 'Здорово!'), ('Wow!', 'Ух ты!'), ('Wow!', 'Ого!'), ('Wow!', 'Вах!'), ('Fire!', 'Огонь!')]
363386


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zа-яё\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_pairs = []

for eng, rus in pairs:
    eng = clean_text(eng)
    rus = clean_text(rus)
    cleaned_pairs.append((eng, rus))

print(cleaned_pairs[:20])

[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здорово'), ('run', 'беги'), ('run', 'бегите'), ('run', 'беги'), ('run', 'бегите'), ('who', 'кто'), ('wow', 'вот это да'), ('wow', 'круто'), ('wow', 'здорово'), ('wow', 'ух ты'), ('wow', 'ого'), ('wow', 'вах'), ('fire', 'огонь')]


In [4]:
processed_pairs = []
for eng, rus in cleaned_pairs:
    rus = "<sos> " + rus + " <eos>"
    processed_pairs.append((eng, rus))

print(processed_pairs[:20])

[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>'), ('hi', '<sos> хай <eos>'), ('hi', '<sos> здрасте <eos>'), ('hi', '<sos> здорово <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>'), ('who', '<sos> кто <eos>'), ('wow', '<sos> вот это да <eos>'), ('wow', '<sos> круто <eos>'), ('wow', '<sos> здорово <eos>'), ('wow', '<sos> ух ты <eos>'), ('wow', '<sos> ого <eos>'), ('wow', '<sos> вах <eos>'), ('fire', '<sos> огонь <eos>')]


In [5]:
def build_vocab(sentences):
    counter = Counter()

    for sent in sentences:
        words = sent.split()
        counter.update(words)

    vocab = {"<pad>": 0, "<oov>": 1}

    for word in counter:
        vocab[word] = len(vocab)

    return vocab

In [6]:
eng_sentences = [p[0] for p in processed_pairs]
rus_sentences = [p[1] for p in processed_pairs]

In [7]:
eng_vocab = build_vocab(eng_sentences)
rus_vocab = build_vocab(rus_sentences)

print("English vocabulary size: ", len(eng_vocab))
print("Russian vocabulary size: ", len(rus_vocab))

English vocabulary size:  16319
Russian vocabulary size:  53282


In [8]:
def sentence_to_idx(sentence, vocab):
    if isinstance(sentence, str):
        sentence = sentence.split()

    return [vocab.get(word, vocab["<oov>"]) for word in sentence]

In [9]:
data = []
for eng, rus in processed_pairs:
    eng_idx = sentence_to_idx(eng, eng_vocab)
    rus_idx = sentence_to_idx(rus, rus_vocab)
    data.append((eng_idx, rus_idx))

print(data[:20])

[([2], [2, 3, 4]), ([2], [2, 5, 4]), ([2], [2, 6, 4]), ([3], [2, 7, 4]), ([3], [2, 8, 4]), ([3], [2, 9, 4]), ([3], [2, 10, 4]), ([3], [2, 11, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([4], [2, 12, 4]), ([4], [2, 13, 4]), ([5], [2, 14, 4]), ([6], [2, 15, 16, 17, 4]), ([6], [2, 18, 4]), ([6], [2, 11, 4]), ([6], [2, 19, 20, 4]), ([6], [2, 21, 4]), ([6], [2, 22, 4]), ([7], [2, 23, 4])]


In [10]:
train_data,val_data = train_test_split(data,test_size=0.2, random_state=42)

print("Training data size: ", len(train_data))
print("Validation data size: ", len(val_data))

Training data size:  290708
Validation data size:  72678


### Data Loaders Creation

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [12]:
class TranslationDataset(Dataset):

    def __init__(self, sentence_pairs):
        self.pairs = sentence_pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        return torch.tensor(eng), torch.tensor(rus)

In [13]:
def collate_fn(batch):
    eng_batch, rus_batch = [], []

    for eng, rus in batch:
        eng_batch.append(eng)
        rus_batch.append(rus)

    eng_padded = pad_sequence(eng_batch, padding_value=0)
    rus_padded = pad_sequence(rus_batch, padding_value=0)

    return eng_padded, rus_padded

In [14]:
train_dataset = TranslationDataset(train_data)
val_dataset = TranslationDataset(val_data)

In [15]:
batch_size = 32

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          collate_fn=collate_fn)

val_loader = DataLoader(val_dataset,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=collate_fn)

In [16]:
for eng_batch, rus_batch in train_loader:
    print(eng_batch.shape)
    print(rus_batch.shape)
    break

torch.Size([16, 32])
torch.Size([15, 32])


### Model Architecture

In [17]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


In [18]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        src_len = encoder_outputs.shape[0]
        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)
        return F.softmax(attention, dim=1)

In [19]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, attention, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim + hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(emb_dim + hidden_dim + hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs, mask=None):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)
        context = torch.bmm(a, encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        output, (hidden, cell) = self.rnn(torch.cat((embedded, context), dim=2), (hidden, cell))
        prediction = self.fc_out(torch.cat((output.squeeze(0), context.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden, cell

In [20]:
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5, mask=None):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs, mask)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[t] if random.random() < teacher_forcing_ratio else top1

        return outputs


In [21]:
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)

### Training

In [22]:
input_dim  = len(eng_vocab)
output_dim = len(rus_vocab)
emb_dim    = 256
hidden_dim = 512
n_layers   = 1
dropout    = 0.5

In [23]:
attention = Attention(hidden_dim)
encoder   = Encoder(input_dim, emb_dim, hidden_dim, n_layers, dropout)
decoder   = Decoder(output_dim, emb_dim, hidden_dim, attention, dropout)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = Seq2Seq(encoder, decoder, device).to(device)
model.apply(init_weights)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(16319, 256)
    (lstm): LSTM(256, 512)
  )
  (decoder): Decoder(
    (attention): Attention(
      (attn): Linear(in_features=1024, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(53282, 256)
    (rnn): LSTM(768, 512)
    (fc_out): Linear(in_features=1280, out_features=53282, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [24]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.0005)
pad_idx   = rus_vocab["<pad>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)


In [25]:
def train(model, loader, optimizer, criterion, clip, device, epoch):
    model.train()
    epoch_loss = 0
    tf_ratio = 0.5

    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)
        optimizer.zero_grad()

        output = model(src, trg, teacher_forcing_ratio=tf_ratio)

        output_dim = output.shape[-1]
        output  = output[1:].view(-1, output_dim)
        trg_flat = trg[1:].view(-1)

        loss = criterion(output, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [26]:
def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg_flat = trg[1:].view(-1)

            loss = criterion(output, trg_flat)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [54]:
clip = 1
num_epochs = 20
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion, clip, device, epoch)
    val_loss   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_att_model_final.pt')
        print(f"  ✓ Best model saved (val loss: {val_loss:.4f})")

    print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

  ✓ Best model saved (val loss: 2.9619)
Epoch 01 | Train: 3.9140 | Val: 2.9619 | LR: 0.000500
  ✓ Best model saved (val loss: 2.4588)
Epoch 02 | Train: 2.1997 | Val: 2.4588 | LR: 0.000500
  ✓ Best model saved (val loss: 2.3143)
Epoch 03 | Train: 1.6382 | Val: 2.3143 | LR: 0.000500
  ✓ Best model saved (val loss: 2.2685)
Epoch 04 | Train: 1.3243 | Val: 2.2685 | LR: 0.000500
Epoch 05 | Train: 1.1245 | Val: 2.2788 | LR: 0.000500
Epoch 06 | Train: 0.9904 | Val: 2.3077 | LR: 0.000500
Epoch 07 | Train: 0.8994 | Val: 2.3231 | LR: 0.000250
Epoch 08 | Train: 0.6760 | Val: 2.2998 | LR: 0.000250
Epoch 09 | Train: 0.5875 | Val: 2.3360 | LR: 0.000250
Epoch 10 | Train: 0.5449 | Val: 2.3581 | LR: 0.000125
Epoch 11 | Train: 0.4374 | Val: 2.3841 | LR: 0.000125
Epoch 12 | Train: 0.3993 | Val: 2.4124 | LR: 0.000125
Epoch 13 | Train: 0.3814 | Val: 2.4351 | LR: 0.000063
Epoch 14 | Train: 0.3280 | Val: 2.4583 | LR: 0.000063
Epoch 15 | Train: 0.3145 | Val: 2.4781 | LR: 0.000063
Epoch 16 | Train: 0.3034 | Val

### Translation

In [45]:
import torch
import torch.nn.functional as F

def translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=5, max_len=50, min_len=3):

    model.eval()

    if isinstance(sentence, str):
        tokens = sentence.lower().split()
    else:
        tokens = sentence

    src_indexes = [eng_vocab.get(t, eng_vocab["<oov>"]) for t in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

    beams = [([rus_vocab["<sos>"]], 0.0, hidden, cell)]

    for step in range(max_len):

        new_beams = []

        for seq, score, h, c in beams:

            last_token = seq[-1]

            if last_token == rus_vocab["<eos>"]:
                new_beams.append((seq, score, h, c))
                continue

            input_tensor = torch.LongTensor([last_token]).to(device)

            with torch.no_grad():
                output, h_new, c_new = model.decoder(input_tensor, h, c, encoder_outputs)

            probs = F.log_softmax(output, dim=1)

            if len(seq) < min_len:
                probs[0][rus_vocab["<eos>"]] = -1e9

            top_probs, top_idx = probs.topk(beam_size)

            for i in range(beam_size):
                token = top_idx[0][i].item()
                token_prob = top_probs[0][i].item()
                new_seq = seq + [token]
                new_score = score + token_prob
                new_beams.append((new_seq, new_score, h_new, c_new))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        if all(seq[-1] == rus_vocab["<eos>"] for seq, _, _, _ in beams):
            break

    best_seq = beams[0][0]
    inv_vocab = {v: k for k, v in rus_vocab.items()}

    result = []
    for idx in best_seq[1:]:
        word = inv_vocab[idx]
        if word == "<eos>":
            break
        result.append(word)

    return result

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq(encoder, decoder, device).to(device)

model.load_state_dict(torch.load("best_att_model_final.pt", map_location=device))

model = model.to(device)

model.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(16319, 256)
    (lstm): LSTM(256, 512)
  )
  (decoder): Decoder(
    (attention): Attention(
      (attn): Linear(in_features=1024, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(53282, 256)
    (rnn): LSTM(768, 512)
    (fc_out): Linear(in_features=1280, out_features=53282, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)

In [55]:
sentences = [
    "hello how are you",
    "i love walking",
    "that is wonderful",
    "this is a good idea",
    "please show me the way",
    "can you help me",
    "i need help",
    "interesting story",
    "that is a difficult question",
    "i will tell you"
]

for sentence in sentences:
    greedy = translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=1)
    beam2  = translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=2)
    beam5  = translate_beam_search(sentence, model, eng_vocab, rus_vocab, device, beam_size=5)

    print("EN:", sentence)
    print("Greedy:", " ".join(greedy))
    print("Beam2:", " ".join(beam2))
    print("Beam5:", " ".join(beam5))
    print("-" * 50)

EN: hello how are you
Greedy: ты с
Beam2: вы с
Beam5: как ты
--------------------------------------------------
EN: i love walking
Greedy: я люблю
Beam2: я люблю
Beam5: я люблю
--------------------------------------------------
EN: that is wonderful
Greedy: чудесно чудесно
Beam2: чудесно чудесно
Beam5: чудесно чудесно
--------------------------------------------------
EN: this is a good idea
Greedy: хорошая идея
Beam2: хорошая идея
Beam5: хорошая идея
--------------------------------------------------
EN: please show me the way
Greedy: пожалуйста дорогу
Beam2: пожалуйста дорогу
Beam5: пожалуйста дорогу
--------------------------------------------------
EN: can you help me
Greedy: ты можешь помочь помочь
Beam2: вы можете помочь помочь
Beam5: вы можете помочь помочь
--------------------------------------------------
EN: i need help
Greedy: помощь помощи
Beam2: помощь помощи
Beam5: помощь помощи
--------------------------------------------------
EN: interesting story
Greedy: история истор